# 02 · Le format CSV

Un collègue veut les courses de taxi de mars 2026 en CSV, « pour pouvoir les ouvrir partout ».
On convertit le fichier Parquet en CSV, puis on regarde ce que l'on gagne et ce que l'on perd.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)
chemin_parquet = DATA / "yellow_tripdata_2026-03.parquet"
chemin_csv = OUTPUT / "courses_2026-03.csv"

## 1. Du Parquet au CSV

On lit le fichier Parquet du mois. `%%time` mesure la durée de la cellule : *Wall time* = durée réelle écoulée.

In [2]:
%%time
courses = pd.read_parquet(chemin_parquet)
courses.shape

CPU times: user 1.03 s, sys: 2.7 s, total: 3.73 s
Wall time: 749 ms


(3952451, 20)

3 952 451 courses et 20 colonnes, lues très vite.

On écrit tout en CSV avec `to_csv` (`index=False` : pas de colonne de numéros de ligne). C'est long : patientez.

In [3]:
%%time
courses.to_csv(chemin_csv, index=False)

CPU times: user 34.4 s, sys: 669 ms, total: 35 s
Wall time: 35.2 s


Plusieurs dizaines de secondes : il faut transformer chaque nombre et chaque date en texte.

Comparons les tailles avec une petite fonction `taille_mo`.

In [4]:
def taille_mo(chemin):
    """Taille d'un fichier en mégaoctets (Mo)."""
    return round(chemin.stat().st_size / 1_000_000, 1)

print("Parquet :", taille_mo(chemin_parquet), "Mo")
print("CSV     :", taille_mo(chemin_csv), "Mo")

Parquet : 67.9 Mo
CSV     : 424.7 Mo


Le CSV pèse environ 425 Mo, plus de 6 fois le Parquet (68 Mo) : chaque valeur est écrite comme du texte, sans compression.

## 2. Un CSV, ce n'est que du texte

On l'ouvre comme un simple fichier texte et on affiche les 3 premières lignes.

In [5]:
with open(chemin_csv, encoding="utf-8") as fichier:
    for i in range(3):
        print(fichier.readline().strip())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2,2026-03-01 00:02:26,2026-03-01 00:13:45,1.0,2.58,1.0,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
2,2026-03-01 00:19:33,2026-03-01 00:28:21,1.0,1.5,1.0,N,238,166,1,10.0,1.0,0.5,0.0,0.0,1.0,15.0,2.5,0.0,0.0


La 1re ligne est l'en-tête (les noms des colonnes), puis une course par ligne, valeurs séparées par des virgules.
Rien ne dit qu'une colonne contient des dates ou des entiers.

## 3. Relire le CSV

On relit le fichier avec `read_csv`.

In [6]:
%%time
courses_csv = pd.read_csv(chemin_csv)

<timed exec>:1: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.


CPU times: user 9.42 s, sys: 1.1 s, total: 10.5 s
Wall time: 10.5 s


Plusieurs secondes, bien plus que pour le Parquet : pandas doit découper le texte et deviner le type de chaque colonne.
Il hésite même sur `store_and_fwd_flag` (`N` ou `Y`, mais vide sur près d'un million de lignes en fin de fichier) : c'est l'avertissement *DtypeWarning*.

Comparons les types des 3 premières colonnes.

In [7]:
types = pd.DataFrame({"Parquet": courses.dtypes, "CSV": courses_csv.dtypes})
types.head(3)

,Parquet,CSV
VendorID,int32,int64
tpep_pickup_datetime,datetime64[us],str
tpep_dropoff_datetime,datetime64[us],str


Les dates sont devenues du texte (`str`) et `VendorID` est passé de `int32` à `int64` : **un CSV n'a pas de types**, pandas les devine.

On donne nous-mêmes les types : les dates avec `parse_dates`, le texte de `store_and_fwd_flag` avec `dtype`.

In [8]:
%%time
dates = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]
courses_csv = pd.read_csv(chemin_csv, parse_dates=dates, dtype={"store_and_fwd_flag": "str"})
courses_csv.dtypes.head(3)

CPU times: user 10.5 s, sys: 771 ms, total: 11.3 s
Wall time: 11.3 s


VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
dtype: object

Les dates sont de nouveau des dates (`datetime64`) et l'avertissement a disparu.
Il faut le préciser à chaque lecture : le fichier, lui, ne connaît pas les types.

## 4. Lire moins

`usecols` ne garde que certaines colonnes ; `nrows` ne lit que les premières lignes.

In [9]:
%%time
distances = pd.read_csv(chemin_csv, usecols=["trip_distance", "total_amount"])
distances.shape

CPU times: user 3.12 s, sys: 137 ms, total: 3.26 s
Wall time: 3.42 s


(3952451, 2)

Plus rapide et bien plus léger en mémoire, mais pandas parcourt quand même tout le fichier : dans un CSV, on ne peut pas sauter directement à une colonne.

In [10]:
pd.read_csv(chemin_csv, nrows=5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-03-01 00:02:26,2026-03-01 00:13:45,1.0,2.58,1.0,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
1,2,2026-03-01 00:19:33,2026-03-01 00:28:21,1.0,1.50,1.0,N,238,166,1,10.0,1.0,0.5,0.00,0.0,1.0,15.00,2.5,0.0,0.00
2,2,2026-03-01 00:07:20,2026-03-01 00:15:12,2.0,0.88,1.0,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75
3,2,2026-03-01 00:16:11,2026-03-01 00:28:20,1.0,1.76,1.0,N,249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75
4,2,2026-03-01 00:20:47,2026-03-01 00:30:44,2.0,1.57,1.0,N,100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75


Instantané : pratique pour jeter un œil à un très gros fichier avant de le charger.

## 5. Lire par paquets

Si le fichier est plus gros que la mémoire, `chunksize` le lit par paquets (*chunks*) de 1 million de lignes : on traite un paquet, puis le suivant.
Ici, on additionne les montants (`total_amount`) de toutes les courses.

In [11]:
total = 0
for paquet in pd.read_csv(chemin_csv, usecols=["total_amount"], chunksize=1_000_000):
    print("Paquet de", len(paquet), "lignes")
    total = total + paquet["total_amount"].sum()
round(total)

Paquet de 1000000 lignes
Paquet de 1000000 lignes
Paquet de 1000000 lignes
Paquet de 952451 lignes


118958457

4 paquets et environ 119 millions de dollars au total, sans jamais charger tout le fichier d'un coup.

## 6. Compresser : `.csv.gz`

Si le nom du fichier finit par `.gz`, pandas compresse tout seul.
La compression est lente : on compare donc CSV et CSV.gz sur les 500 000 premières courses.

In [12]:
%%time
extrait = courses.head(500_000)
chemin_extrait = OUTPUT / "courses_2026-03_extrait.csv"
extrait.to_csv(chemin_extrait, index=False)

CPU times: user 5.54 s, sys: 98.9 ms, total: 5.64 s
Wall time: 5.65 s


In [13]:
%%time
chemin_gz = OUTPUT / "courses_2026-03_extrait.csv.gz"
extrait.to_csv(chemin_gz, index=False)

CPU times: user 16.2 s, sys: 18.2 ms, total: 16.3 s
Wall time: 16.3 s


Même contenu, mais l'écriture compressée prend environ 3 fois plus de temps.

In [14]:
print("CSV    :", taille_mo(chemin_extrait), "Mo")
print("CSV.gz :", taille_mo(chemin_gz), "Mo")

CSV    : 55.3 Mo
CSV.gz : 9.4 Mo


Environ 6 fois plus petit (55 Mo → 9 Mo) : idéal pour transporter un CSV (envoi, téléchargement).

In [15]:
pd.read_csv(chemin_gz).shape

(500000, 20)

Relu sans aucune option : pandas reconnaît l'extension `.gz` et décompresse tout seul.

## 7. Piège n° 1 : le séparateur

Excel en français exporte avec `;` comme séparateur et la virgule comme décimale (2,5).
On crée un petit fichier de commandes de la boutique du cours (à Cotonou) écrit de cette façon.

In [16]:
chemin_fr = OUTPUT / "commandes_fr.csv"
chemin_fr.write_text("""commande_id;date_commande;quartier;telephone;montant_fcfa;poids_kg
CMD001;2026-10-01;Gbégamey;0197000001;45000;2,5
CMD002;2026-10-02;Akpakpa;0196000002;12500;0,8
CMD003;2026-10-03;Cadjèhoun;0195000004;8000;1,2
""", encoding="utf-8")

pd.read_csv(chemin_fr)

,commande_id;date_commande;quartier;telephone;montant_fcfa;poids_kg
CMD001;2026-10-01;Gbégamey;0197000001;45000;2,5
CMD002;2026-10-02;Akpakpa;0196000002;12500;0,8
CMD003;2026-10-03;Cadjèhoun;0195000004;8000;1,2


Raté : pandas attend des virgules. Tout l'en-tête devient un seul nom de colonne, et la virgule de « 2,5 » coupe chaque ligne en deux.

On annonce le séparateur et la décimale.

In [17]:
commandes = pd.read_csv(chemin_fr, sep=";", decimal=",")
commandes

,commande_id,date_commande,quartier,telephone,montant_fcfa,poids_kg
0,CMD001,2026-10-01,Gbégamey,197000001,45000,2.5
1,CMD002,2026-10-02,Akpakpa,196000002,12500,0.8
2,CMD003,2026-10-03,Cadjèhoun,195000004,8000,1.2


6 colonnes, et `poids_kg` contient de vrais nombres (2.5).
Mais regardez `telephone` : le 0 du début a disparu (voir le piège n° 3).

## 8. Piège n° 2 : l'encodage

Un vieux logiciel comptable sous Windows exporte en `cp1252`, pas en UTF-8. On fabrique un tel fichier et on regarde ses octets.

In [18]:
chemin_compta = OUTPUT / "export_compta.csv"
chemin_compta.write_text("""commande_id,quartier,montant_fcfa,poids_kg
CMD001,Gbégamey,45000,2.5
CMD003,Cadjèhoun,8000,1.2
""", encoding="cp1252")
chemin_compta.read_bytes()

b'commande_id,quartier,montant_fcfa,poids_kg\nCMD001,Gb\xe9gamey,45000,2.5\nCMD003,Cadj\xe8houn,8000,1.2\n'

En cp1252, « é » et « è » tiennent sur un seul octet (`\xe9`, `\xe8`) ; en UTF-8, il en faut deux (« é » = `\xc3\xa9`).

On lit ce fichier sans rien préciser.

In [19]:
try:
    pd.read_csv(chemin_compta)
except UnicodeDecodeError as erreur:
    print("Erreur :", erreur)

Erreur : 'utf-8' codec can't decode byte 0xe9 in position 52: invalid continuation byte


pandas lit en UTF-8 par défaut : en UTF-8, l'octet `0xe9` seul ne veut rien dire, d'où l'erreur.

On annonce l'encodage.

In [20]:
compta = pd.read_csv(chemin_compta, encoding="cp1252")
compta

,commande_id,quartier,montant_fcfa,poids_kg
0,CMD001,Gbégamey,45000,2.5
1,CMD003,Cadjèhoun,8000,1.2


Les accents sont corrects.

Si on ne connaît pas l'encodage, la bibliothèque `charset_normalizer` peut essayer de le deviner.

In [21]:
from charset_normalizer import from_path

from_path(chemin_compta).best().encoding

'cp1250'

Ici, elle répond `cp1250` (Europe centrale) : proche, mais avec lui « Cadjèhoun » deviendrait « Cadjčhoun ».
C'est une devinette, pas une preuve : demandez l'encodage à celui qui vous envoie le fichier.

Pour la compta, qui ouvre tout dans Excel en français, on écrit avec `;`, la virgule décimale et `utf-8-sig` (UTF-8 avec une signature au début, le *BOM*, qu'Excel reconnaît).

In [22]:
chemin_excel = OUTPUT / "pour_la_compta.csv"
compta.to_csv(chemin_excel, index=False, sep=";", decimal=",", encoding="utf-8-sig")
chemin_excel.read_bytes()

b'\xef\xbb\xbfcommande_id;quartier;montant_fcfa;poids_kg\nCMD001;Gb\xc3\xa9gamey;45000;2,5\nCMD003;Cadj\xc3\xa8houn;8000;1,2\n'

Le fichier commence par le BOM (`\xef\xbb\xbf`), « é » tient sur 2 octets (`\xc3\xa9`), les séparateurs sont des `;` et les poids s'écrivent `2,5`.
Excel en français l'ouvrira correctement.

## 9. Piège n° 3 : les types

Le téléphone a perdu son 0 (lu comme un nombre) et la date est lue comme du texte.
On impose les types à la lecture avec `dtype` et `parse_dates`.

In [23]:
commandes = pd.read_csv(
    chemin_fr, sep=";", decimal=",",
    dtype={"telephone": "string"}, parse_dates=["date_commande"],
)
commandes

,commande_id,date_commande,quartier,telephone,montant_fcfa,poids_kg
0,CMD001,2026-10-01,Gbégamey,0197000001,45000,2.5
1,CMD002,2026-10-02,Akpakpa,0196000002,12500,0.8
2,CMD003,2026-10-03,Cadjèhoun,0195000004,8000,1.2


In [24]:
commandes.dtypes

commande_id                 str
date_commande    datetime64[us]
quartier                    str
telephone                string
montant_fcfa              int64
poids_kg                float64
dtype: object

Le téléphone garde son 0 (type `string`) et `date_commande` est une vraie date (`datetime64`) : on peut trier ou filtrer par date.

## Exercices

### ✍️ Exercice 1 — Le pourboire des paiements par carte

Lisez seulement 3 colonnes du gros CSV (`payment_type`, `tip_amount`, `total_amount`).
Calculez le pourboire moyen et le montant moyen des courses payées par carte (`payment_type == 1`).

In [25]:
# ✅ Solution
pourboires = pd.read_csv(chemin_csv, usecols=["payment_type", "tip_amount", "total_amount"])
carte = pourboires[pourboires["payment_type"] == 1]
carte[["tip_amount", "total_amount"]].mean().round(2)

tip_amount       4.16
total_amount    29.87
dtype: float64

En moyenne 4,16 $ de pourboire pour une course à 29,87 $ payée par carte (environ 14 %).
On se limite à la carte car les pourboires en espèces ne sont pas enregistrés dans ces données.

### ✍️ Exercice 2 — Un export pour Excel en français

Exportez les 1 000 premières courses « pour Excel en français » dans `output/exo02_courses_excel_fr.csv`, puis relisez le fichier correctement.

In [26]:
# ✅ Solution
chemin_exo = OUTPUT / "exo02_courses_excel_fr.csv"
courses.head(1000).to_csv(chemin_exo, index=False, sep=";", decimal=",", encoding="utf-8-sig")
relu = pd.read_csv(chemin_exo, sep=";", decimal=",", encoding="utf-8-sig")
relu.head(3)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-03-01 00:02:26,2026-03-01 00:13:45,1.0,2.58,1.0,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
1,2,2026-03-01 00:19:33,2026-03-01 00:28:21,1.0,1.50,1.0,N,238,166,1,10.0,1.0,0.5,0.00,0.0,1.0,15.00,2.5,0.0,0.00
2,2,2026-03-01 00:07:20,2026-03-01 00:15:12,2.0,0.88,1.0,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75


Les montants sont relus comme de vrais nombres (24.94) : même « contrat » à l'écriture et à la lecture (`;`, `,`, `utf-8-sig`).

## À retenir

- Le CSV est du texte : lisible partout, mais environ 6 fois plus gros que le Parquet et bien plus lent à relire.
- Un CSV n'a pas de types : imposez-les à la lecture (`parse_dates`, `dtype={"telephone": "string"}`).
- Gros fichier : `usecols`, `nrows`, `chunksize` ; pour le transporter : `.csv.gz`.
- Annoncez toujours séparateur, décimale et encodage ; pour Excel en français : `sep=";"`, `decimal=","`, `encoding="utf-8-sig"`.